# Example 1: 使用Modular Strategy执行一个Job

In [1]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()

## 修改inp, 并从该Job的Odb中提取信息

In [2]:
spec1 = JobSpec(
	job_name = "planar_stress_odb",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

import pprint
pprint.pprint(spec1)

JobSpec(job_name='planar_stress_odb',
        workflow='modular',
        preparation=PreparationSpec(kind='inp_based',
                                    source_path='./examples/cae_file/planar_stress_template.inp',
                                    params={'load_magnitude': 2000,
                                            'youngs_modulus': 210000},
                                    options={}),
        preflight=None,
        monolithic_script=None,
        monolithic_params={},
        pre_extraction=[],
        post_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_max_stress_mises.py',
                                  tasks=[{'result_name': 'max_stress_mises'},
                                         {'result_name': 'max_displacement'}])],
        subroutine=None,
        meta={})


In [3]:
processor_odb = BatchAbaqusProcessor(
	batch_data = [spec1],
	base_output_dir = os.path.join(CWD, "examples/01_SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes_odb = processor_odb.run_batch(
	num_parallel_jobs = 4,
)

Output()

In [5]:
outcomes_odb

[JobOutcome(job_name='planar_stress_odb', status='COMPLETED', results={'max_stress_mises': 4525.26025390625, 'max_displacement': 3.9895615577697754}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/01_SingleParameterizedJob/output\\planar_stress_odb', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1785912700.6829748, 'ended_at': 1785912700.6849744, 'duration_s': 0.0019996166229248047, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1785912700.6849744, 'ended_at': 1785912757.6265228, 'duration_s': 56.941548347473145, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1785912757.6265228, 'ended_at': 1785912760.176402, 'duration_s': 2.549879312515259, 'error': None}], duration_s=59.887511014938354)]

In [6]:
for oc in outcomes_odb:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['max_stress_mises'])
		print(oc.job_name, oc.results['max_displacement'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

# JobOutcome 自带 job_name / status / results / error / diagnostics,
# 需要按名字索引时直接建字典即可
results_by_name = {oc.job_name: oc for oc in outcomes_odb}

planar_stress_odb 4525.26025390625
planar_stress_odb 3.9895615577697754


## 修改inp, 并从inp中提取信息

In [7]:
spec2 = JobSpec(
	job_name = "planar_stress_mass",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	]
)

In [8]:
processor_mass = BatchAbaqusProcessor(
	batch_data = [spec2],
	base_output_dir = os.path.join(CWD, "examples/01_SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [9]:
outcomes_mass = processor_mass.run_batch(
	num_parallel_jobs = 4,
)

Output()

In [10]:
outcomes_mass

[JobOutcome(job_name='planar_stress_mass', status='COMPLETED', results={'total_mass': 0.00032066262255247625}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/01_SingleParameterizedJob/output\\planar_stress_mass', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1785912904.8555725, 'ended_at': 1785912904.8570776, 'duration_s': 0.001505136489868164, 'error': None}, {'phase': 'pre_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1785912904.8570776, 'ended_at': 1785912908.4154868, 'duration_s': 3.5584092140197754, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1785912908.4154868, 'ended_at': 1785912962.743768, 'duration_s': 54.32828116416931, 'error': None}], duration_s=58.25226664543152)]

In [11]:
for oc in outcomes_mass:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['total_mass'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

# outcomes_mass 本身就是 list[JobOutcome], 不需要再转换

planar_stress_mass 0.00032066262255247625
